In [109]:
import pandas as pd
from pathlib import Path
from urllib.parse import urlparse
import re

In [110]:
# 1. Define Paths Robustly (Works in both Jupyter and standard scripts)
current_dir = Path.cwd()

# Handle path depending on whether you run this from root or inside the 'scripts' folder
if current_dir.name == "scripts":
    root_dir = current_dir.parent
else:
    root_dir = current_dir

seller_data_dir = root_dir / 'data' / 'csvs'
seller_name = 'Arogga'
data_file = f"{seller_name}_products.csv"
data_file_path = seller_data_dir / data_file


# Upload directory
upload_dir = root_dir / 'data' / 'csvs' / 'uploads' / seller_name
upload_dir.mkdir(parents=True, exist_ok=True)


In [111]:
# Define some helper functions for cleaning
def slugify(text):
    if isinstance(text, str):
        # Remove anything that is not alphanumeric or space, then replace spaces with hyphens and convert to lowercase
        text = ''.join(e for e in text if e.isalnum() or e.isspace())
        text = text.replace(' ', '-').lower()
    return text

def extract_base_url(url):
    """Extracts the base url (e.g., https://www.startech.com.bd/) from a full product link."""
    if pd.isna(url) or not isinstance(url, str):
        return ""
    parsed = urlparse(url)
    return f"{parsed.scheme}://{parsed.netloc}/"

def strip_lines(text):
    """Removes newline characters from text."""
    if isinstance(text, str):
        return text.replace('\n', ' ').strip()
    return text

BASE_PRODUCT = {
    "category_slug": "", "product_name": "", "product_slug": "", "seller_slug": "",
    "current_price": "", "seller_product_url": "", "brand_slug": "", "brand_name": "", 
    "product_description": "", "model": "", "sku": "", "primary_image_url": "", 
    "image_urls": "", "specifications": "", "attributes": "", "variation_type": "",
    "parent_product_slug": "", "original_price": "", "currency": "BDT", 
    "in_stock": "YES", "stock_quantity": "", "seller_rating": "", "review_count": "", 
    "shipping_cost": "", "free_shipping": "", "estimated_delivery_days": "", 
    "seller_sku": "", "seller_product_name": "", "category_path": "", 
    "category_name": "", "category_description": "", "seller_name": "",
    "base_url": "", "seller_country_code": "BD", "is_active": ""
}

In [112]:
pd.set_option('display.max_columns', None)  # Show all columns when printing DataFrames
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)  # Set display width to avoid line breaks in the middle of rows
pd.set_option('display.max_colwidth', None)  # Show full content of each cell without truncation


In [113]:
df = pd.read_csv(data_file_path)

# Remove any 'Unnamed' columns caused by trailing commas in the CSV
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

print(f"Loaded {data_file_path} - Shape: {df.shape}")
print("Columns:", df.columns.tolist())
print(df.info())

Loaded /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/Arogga_products.csv - Shape: (8818, 12)
Columns: ['id', 'title', 'sale_price', 'link', 'price', 'primary_image_url', 'image_urls', 'stock', 'brand', 'description', 'rating', 'reviews']
<class 'pandas.DataFrame'>
RangeIndex: 8818 entries, 0 to 8817
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 8818 non-null   int64  
 1   title              8818 non-null   str    
 2   sale_price         8818 non-null   float64
 3   link               8818 non-null   str    
 4   price              8818 non-null   float64
 5   primary_image_url  8810 non-null   str    
 6   image_urls         8810 non-null   str    
 7   stock              8818 non-null   int64  
 8   brand              8809 non-null   str    
 9   description        8140 non-null   str    
 10  rating             8807 non-null   float64
 11  reviews            8807 non-null 

In [114]:

transformed_df = pd.DataFrame()

transformed_df['product_name'] = df['title']
transformed_df['product_slug'] = df['title'].fillna('').apply(slugify)
transformed_df['product_description'] = df['description'].fillna('').apply(strip_lines)
transformed_df['seller_product_name'] = df['title']

transformed_df['current_price'] = df['sale_price']
transformed_df['original_price'] = df['price']
transformed_df['seller_product_url'] = df['link']
transformed_df['brand_name'] = df['brand'].fillna('')

transformed_df['category_name'] = 'Uncategorized'
transformed_df['category_slug'] = 'uncategorized'

transformed_df['primary_image_url'] = df['primary_image_url'].fillna('')
transformed_df['image_urls'] = df['image_urls'].fillna('').apply(strip_lines)
transformed_df['review_count'] = df['reviews'].fillna(0).astype(int)

transformed_df['stock_quantity'] = df['stock'].fillna(0).astype(int)
transformed_df['in_stock'] = df['stock'] > 0
transformed_df['is_active'] = transformed_df['in_stock']

transformed_df['seller_name'] = seller_name
transformed_df['seller_rating'] = df['rating']
transformed_df['seller_slug'] = slugify(seller_name)
transformed_df['base_url'] = df['link'].apply(extract_base_url)

transformed_df['currency'] = 'BDT'
transformed_df['seller_country_code'] = 'BD'

transformed_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 8818 entries, 0 to 8817
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_name         8818 non-null   str    
 1   product_slug         8818 non-null   str    
 2   product_description  8818 non-null   str    
 3   seller_product_name  8818 non-null   str    
 4   current_price        8818 non-null   float64
 5   original_price       8818 non-null   float64
 6   seller_product_url   8818 non-null   str    
 7   brand_name           8818 non-null   str    
 8   category_name        8818 non-null   str    
 9   category_slug        8818 non-null   str    
 10  primary_image_url    8818 non-null   str    
 11  image_urls           8818 non-null   str    
 12  review_count         8818 non-null   int64  
 13  stock_quantity       8818 non-null   int64  
 14  in_stock             8818 non-null   bool   
 15  is_active            8818 non-null   bool   
 16 

In [115]:
# Loop through the base schema. If a column hasn't been created yet, fill it with the default empty value
for col, default_val in BASE_PRODUCT.items():
    if col not in transformed_df.columns:
        transformed_df[col] = None

# Rearrange columns to match the BASE_PRODUCT schema
transformed_df = transformed_df[list(BASE_PRODUCT.keys())]

transformed_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8818 entries, 0 to 8817
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   category_slug            8818 non-null   str    
 1   product_name             8818 non-null   str    
 2   product_slug             8818 non-null   str    
 3   seller_slug              8818 non-null   str    
 4   current_price            8818 non-null   float64
 5   seller_product_url       8818 non-null   str    
 6   brand_slug               0 non-null      object 
 7   brand_name               8818 non-null   str    
 8   product_description      8818 non-null   str    
 9   model                    0 non-null      object 
 10  sku                      0 non-null      object 
 11  primary_image_url        8818 non-null   str    
 12  image_urls               8818 non-null   str    
 13  specifications           0 non-null      object 
 14  attributes               0 non-null

In [116]:
# Make df = transformed_df for easier handling in the next steps
df = transformed_df.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8818 entries, 0 to 8817
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   category_slug            8818 non-null   str    
 1   product_name             8818 non-null   str    
 2   product_slug             8818 non-null   str    
 3   seller_slug              8818 non-null   str    
 4   current_price            8818 non-null   float64
 5   seller_product_url       8818 non-null   str    
 6   brand_slug               0 non-null      object 
 7   brand_name               8818 non-null   str    
 8   product_description      8818 non-null   str    
 9   model                    0 non-null      object 
 10  sku                      0 non-null      object 
 11  primary_image_url        8818 non-null   str    
 12  image_urls               8818 non-null   str    
 13  specifications           0 non-null      object 
 14  attributes               0 non-null

In [117]:
# Check the brand_name column for any anomalies
print("\nUnique brands in 'brand_name':")
print(df['brand_name'].value_counts())


Unique brands in 'brand_name':
brand_name
Loreal                                      205
Dove                                        176
Swiss Beauty                                152
LAIKOU                                      129
Sunsilk                                     114
Square Toiletries Limited                   112
Nivea                                       110
Beauty Glazed                               106
Marico Bangladesh                           101
Tresemme                                    101
Garnier                                      99
Vaseline                                     84
VGR                                          77
Rongon Herbals                               77
No Brand                                     70
Insight                                      68
SHEGLAM                                      64
Global Pharma Distributors Ltd               61
Fogg                                         61
Lux                                          

In [118]:
# Create a new dataframe containing brand_name, seller_product_name, seller_product_url for 2 rows for each unique brand
print("\nSample products for each brand:")
# Collect unique brands
unique_brands = df['brand_name'].unique()
combined_unique_brand_df = pd.DataFrame()
for brand in unique_brands:
    # Append 2 elements to the brand_df for each brand
    brand_df = df[df['brand_name'] == brand][['brand_name', 'seller_product_name', 'seller_product_url']]
    combined_unique_brand_df = pd.concat([combined_unique_brand_df, brand_df.head(2)], ignore_index=True)
print("\nCombined dataframe for each brand:")
print(combined_unique_brand_df)

# Save the combined_unique_brand_df to a CSV file for review
review_file_path = upload_dir / f"{seller_name}_brand_review.csv"
combined_unique_brand_df.to_csv(review_file_path, index=False)
print(f"\nSaved brand review file to: {review_file_path}")



Sample products for each brand:

Combined dataframe for each brand:
                                    brand_name                                                                                                                                                                         seller_product_name                                                                                                                                                                                                       seller_product_url
0                                       Axis-Y                                                                                                                                                  AXIS-Y Dark Spot Correcting Glow Serum 5ml                                                                                                                                          https://www.arogga.com/product/55623/axis-y-dark-spot-correcting-glow-serum-5ml
1                          

In [119]:
def fix_brand_name(row):
    brand = str(row['brand_name']).strip()
    product = str(row['seller_product_name']).strip()
    
    # 1. Check if the brand is missing or flagged as "No Brand"
    is_invalid = pd.isna(row['brand_name']) or brand.lower() in['nan', 'no brand', ''] or brand == ','
    
    # 2. Check if the brand is actually a Corporate/Distributor name
    corp_keywords =[
        'ltd', 'limited', 'corp', 'corporation', 'pvt', 'llp', 'gmbh', 
        'l.l.c', 'llc', 'pharma', 'pharmaceuticals', 'pharmaceutica', 
        'laboratories', 'laboratoire', 'distributors', 'trading', 
        'trade', 'international', 'plc'
    ]
    # Create a regex pattern to match whole words only
    corp_pattern = r'\b(?:' + '|'.join(corp_keywords) + r')\b'
    is_corporate = bool(re.search(corp_pattern, brand.lower()))
    
    # If the brand looks valid, keep it exactly as it is
    if not (is_invalid or is_corporate):
        return brand
        
    # --- STRATEGY: Extract from Product Name ---
    
    if pd.isna(product) or not product:
        return "Unknown"
        
    # Clean promotional text from the beginning of the title (e.g., "Buy 1 ", "Buy 2 ")
    clean_product = re.sub(r'^Buy\s+\d+\s+', '', product, flags=re.IGNORECASE)
    
    words = clean_product.split()
    if not words:
        return "Unknown"
        
    first_word = words[0]
    
    # Clean trailing punctuation from the extracted word (e.g., "Mederma," -> "Mederma")
    first_word = re.sub(r'[,:-]$', '', first_word)
    
    # If the first word is a title, take the first TWO words (e.g., "Dr. Rashel", "The Ordinary")
    titles =['the', 'dr.', 'dr', 'mr.', 'mr', 'miss', 'mrs.']
    if first_word.lower() in titles and len(words) > 1:
        second_word = re.sub(r'[,:-]$', '', words[1])
        return f"{first_word} {second_word}"
        
    return first_word

# Apply the function to create a new corrected column (or overwrite the existing one)
df['corrected_brand_name'] = df.apply(fix_brand_name, axis=1)

# # Display a sample of the fixes to verify
# fixes = df[df['brand_name'] != df['corrected_brand_name']][['brand_name', 'seller_product_name', 'corrected_brand_name']]
# print(fixes.head(20))

df['corrected_brand_name'].value_counts()

corrected_brand_name
Loreal                            205
Dove                              176
Swiss Beauty                      152
LAIKOU                            129
Sunsilk                           114
Nivea                             110
Beauty Glazed                     106
Marico Bangladesh                 101
Tresemme                          101
Garnier                            99
Vaseline                           84
VGR                                77
Rongon Herbals                     77
Insight                            69
SHEGLAM                            64
Fogg                               61
Lux                                60
Cerave                             59
Imagic                             59
Skin'O                             58
Himalaya Bangladesh                57
Reckitt Benckiser Bangladesh       57
Technic                            56
Golden Girl                        55
Enchanteur                         54
Pantene                      

In [120]:
# Put the corrected brand name back into the main brand_name column
df['brand_name'] = df['corrected_brand_name']
df.drop(columns=['corrected_brand_name'], inplace=True)

# Make the 'brand_slug' column by slugifying the cleaned 'brand_name'
df['brand_slug'] = df['brand_name'].apply(slugify)

# Show the df info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8818 entries, 0 to 8817
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   category_slug            8818 non-null   str    
 1   product_name             8818 non-null   str    
 2   product_slug             8818 non-null   str    
 3   seller_slug              8818 non-null   str    
 4   current_price            8818 non-null   float64
 5   seller_product_url       8818 non-null   str    
 6   brand_slug               8818 non-null   str    
 7   brand_name               8818 non-null   str    
 8   product_description      8818 non-null   str    
 9   model                    0 non-null      object 
 10  sku                      0 non-null      object 
 11  primary_image_url        8818 non-null   str    
 12  image_urls               8818 non-null   str    
 13  specifications           0 non-null      object 
 14  attributes               0 non-null

In [ ]:
# Save the csv file to the csvs folder with the name "{seller_name}_products_cleaned.csv"
cleaned_data_file = f"{seller_name}_products_cleaned.csv"
cleaned_data_file_path = seller_data_dir / cleaned_data_file
df.to_csv(cleaned_data_file_path, index=False)
print(f"\nCleaned data saved to {cleaned_data_file_path}")


Cleaned data saved to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/Arogga_products_cleaned.csv


In [122]:
# Put the cleaned data into 2500 rows per csv file into the uploads directory
chunk_size = 2500
for i in range(0, len(df), chunk_size):
    chunk = df[i:i+chunk_size]
    chunk_file_path = upload_dir / f"{seller_name}_products_cleaned_part_{i//chunk_size + 1}.csv"
    chunk.to_csv(chunk_file_path, index=False)
    print(f"Saved chunk {i//chunk_size + 1} to {chunk_file_path}")

Saved chunk 1 to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/uploads/Arogga/Arogga_products_cleaned_part_1.csv
Saved chunk 2 to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/uploads/Arogga/Arogga_products_cleaned_part_2.csv
Saved chunk 3 to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/uploads/Arogga/Arogga_products_cleaned_part_3.csv
Saved chunk 4 to /home/abrar/Freelance/DaamDekho/Crawler/data/csvs/uploads/Arogga/Arogga_products_cleaned_part_4.csv
